## Internet2 Benchmarks
This notebook includes the properties which were verified as a part of the comparison to prior works and discussed in this directory's [README](../README.md). Those properties are the _BlockToExternal_ and _NoMartians_ properties.

* The _No Martians_ property verifies that no routers in the network accept any martain prefixes (a predetermined set of bad prefixes).
* The _Block to External_ property verifies that no router advertises routes with a particular commmunity to neighbors outside the network.

In [1]:
# %run api/startup.py
import sys,os
# sys.path.insert(0, '../api')
import api.whirlpool,api.startup
%run api/startup.py

In [2]:
from api.startup import runIsolationVerificationQuestion,runWhirlpoolQuery,create
from api.whirlpool import negate, trueAtLocation, outgoingEdgesFrom, incomingEdgesTo
from api.whirlpool import LocationPropertyPair, IsolationQuery, Clause, OUTGOING
from common import counterexamplesCount,FULL_SNAPSHOT,INDIVIDUAL_SNAPS,BTE_Community,MyTimer,ROUTERS,MARTIANS

### Block to External Property

In [3]:
# Create the Batfish Session for the BlockToExternal tests
networkName = "BlockToExternal_Check_net"
snapshotName = "BlockToExternal_Check_snapshot"
bf = create(networkName,snapshotName,FULL_SNAPSHOT)

In [4]:
# Runs the verification query without necessary assumptions and prints runtime
target = LocationPropertyPair(location=OUTGOING,property=[Clause(communities=[negate(BTE_Community)])])
query = IsolationQuery(target=target,compute_dp=True)
t = MyTimer()
df = runIsolationVerificationQuestion(bf,query)
elapsed = t.stop()

print(f"Runtime: {elapsed} seconds")
print(f"Counterexample rows: {counterexamplesCount(df)}")

Runtime: 36.37 seconds
Counterexample rows: 84


In [5]:
# Displays the data frame with the verification details, this will be long because all invariants are displayed
show(df)

,Location_Relevance,Provided_Invariant,Network_Locations,Inferred_Invariant,Counterexample
0,Target,!comm(11537:888),ALL-OUTGOING,!comm(11537:888),
1,Assumption,true,"198.71.45.211 -> 64.57.28.249 (wash), 200.0.207.9 -> 64.57.28.243 (atla-re0)",LIMIT (Complex BDD) [41],"Bgpv4Route{network=8.8.8.0/24, asPath=[17579, 27750, 5663, 6360], communities=CommunitySet{communities=[11537:888]}}"
2,Assumption,true,"64.57.24.196 -> 64.57.28.244 (hous), 64.57.24.217 -> 64.57.28.243 (atla-re0), 64.57.24.197 -> 64.57.28.244 (hous), 64.57.25.165 -> 64.57.28.243 (atla-re0), 64.57.25.110 -> 64.57.28.243 (atla-re0), 64.57.25.116 -> 64.57.28.243 (atla-re0), 64.57.25.110 -> 64.57.28.241 (chic), 64.57.25.164 -> 64.57.28.249 (wash), 64.57.24.197 -> 64.57.28.243 (atla-re0), 64.57.25.164 -> 64.57.28.243 (atla-re0), 64.57.25.116 -> 64.57.28.244 (hous), 64.57.24.196 -> 64.57.28.243 (atla-re0), 64.57.25.108 -> 64.57.28.243 (atla-re0), 64.57.25.108 -> 64.57.28.241 (chic), 64.57.25.165 -> 64.57.28.249 (wash), 64.57.24.217 -> 64.57.28.244 (hous)",LIMIT (Complex BDD) [6],"Bgpv4Route{network=64.57.31.0/24, asPath=[27750, 174, 17579, 19401, 6360, 64512], communities=CommunitySet{communities=[11537:888]}}"
3,Assumption,true,192.31.99.133 -> 64.57.28.241 (chic),LIMIT (Complex BDD) [53],"Bgpv4Route{network=8.8.8.0/24, asPath=[27750], communities=CommunitySet{communities=[11537:888]}}"
4,Assumption,true,"205.233.255.36 -> 64.57.28.243 (atla-re0), 205.233.255.32 -> 64.57.28.244 (hous)",LIMIT (Complex BDD) [65],"Bgpv4Route{network=143.132.0.0/16, asPath=[17579, 27750, 5663, 6360], communities=CommunitySet{communities=[11537:888]}}"
5,Assumption,true,"216.27.100.5 -> 64.57.28.242 (newy-re0), 216.27.100.17 -> 64.57.28.249 (wash)",LIMIT (Complex BDD) [21],"Bgpv4Route{network=129.32.0.0/16, asPath=[17579, 27750, 5663, 6360], communities=CommunitySet{communities=[11537:888]}}"
6,Assumption,true,198.71.46.77 -> 64.57.28.241 (chic),LIMIT (Complex BDD) [77],"Bgpv4Route{network=192.35.252.0/24, asPath=[17579, 27750, 5663, 6360], communities=CommunitySet{communities=[11537:888]}}"
7,Assumption,true,10.11.1.17 -> 64.57.28.242 (newy-re0),LIMIT (Complex BDD) [33],"Bgpv4Route{network=115.114.246.16/32, asPath=[27750, 174, 17579, 19401, 6360, 64512]}"
8,Assumption,true,"198.71.46.154 -> 64.57.28.249 (wash), 198.71.46.152 -> 64.57.28.250 (clev-re0)",LIMIT (Complex BDD) [81],"Bgpv4Route{network=130.14.0.0/16, asPath=[17579, 27750, 5663, 6360], communities=CommunitySet{communities=[11537:888]}}"
9,Assumption,true,130.111.0.84 -> 64.57.28.250 (clev-re0),LIMIT (Complex BDD) [61],"Bgpv4Route{network=169.244.0.0/16, asPath=[17579, 27750, 5663, 6360], communities=CommunitySet{communities=[11537:888]}}"


In [6]:
# Runs the verification query which isolates which outgoing edges violate the intended BlockToExternal property
from block_to_external import isoViolations

target = LocationPropertyPair(location=OUTGOING,property=[Clause(communities=[negate(BTE_Community)])])
query = IsolationQuery(target=target,isolate_violations=True,compute_dp=True)
t = MyTimer()
df = runIsolationVerificationQuestion(bf,query)
elapsed = t.stop()

print(f"Runtime: {elapsed} seconds")
print(isoViolations(df)) 

Runtime: 118.9 seconds
[ALL OTHER TARGETS INDEPENDENTLY VERIFY] INDEPENDENT VIOLATIONS AT: (chic) 64.57.28.241 -> 207.75.164.213, (chic) 64.57.28.241 -> 207.75.164.233, (newy-re0) 64.57.28.242 -> 10.11.1.17, (chic) 64.57.28.241 -> 128.223.51.108, (losa) 64.57.28.248 -> 203.181.248.35, (chic) 64.57.28.241 -> 128.223.51.102


In [7]:
# Runs the verification query with necessary assumptions and prints runtime (should show no counterexample rows)
violations = ["chic -> 207.75.164.233", "chic -> 128.223.51.108", "chic -> 128.223.51.102", 
    "chic -> 207.75.164.213", "losa -> 203.181.248.35", "newy-re0 -> 10.11.1.17"]
target = LocationPropertyPair(location=OUTGOING,property=[Clause(communities=[negate(BTE_Community)])])
query = IsolationQuery(target=target,assumptions=list(map(trueAtLocation,violations)),compute_dp=True)
t = MyTimer()
df = runIsolationVerificationQuestion(bf,query)
elapsed = t.stop()

print(f"Runtime: {elapsed} seconds")
print(f"Counterexample rows: {counterexamplesCount(df)}")

Runtime: 37.37 seconds
Counterexample rows: 0


In [8]:
# Displays the data frame with the verification details, this will be long because all invariants are displayed
show(df)

Location_Relevance Provided_Invariant  \
0             Target   !comm(11537:888)   
1         Assumption               true   
2  Internal Location                      
3  Internal Location               true   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [9]:
# Runs verification to show that no router strips the BlockToExternal community
routers = [("atla","atla-re0"),("chic","chic"),("clev","clev-re0"),("hous","hous"),("kans","kans-re0"),
           ("losa","losa"),("newy32aoa","newy-re0"),("salt","salt-re0"),("seat","seat-re0"),("wash","wash")]
for (directory,node) in routers:
    path = f"{INDIVIDUAL_SNAPS}/{directory}"
    target = LocationPropertyPair(location=outgoingEdgesFrom(node),property=[Clause(communities=[BTE_Community])])
    assumptions = [LocationPropertyPair(location=incomingEdgesTo(node),property=[Clause(communities=[BTE_Community])])]
    query = IsolationQuery(target=target,assumptions=assumptions)
    t = MyTimer()
    df = runWhirlpoolQuery(networkName,snapshotName,path,query)
    elapsed = t.stop()

    noStrip = True
    for k in df.itertuples():
        if k.Counterexample.strip() != "":
            noStrip = False
            break
    
    if noStrip:
        print(f"[{node}] does not strip bte community ({elapsed} s)")
    else:
        print(f"[{node}] DOES STRIP bte community ({elapsed} s)")

[atla-re0] does not strip bte community (0.85 s)
[chic] does not strip bte community (1.92 s)
[clev-re0] does not strip bte community (0.62 s)
[hous] does not strip bte community (0.61 s)
[kans-re0] does not strip bte community (0.6 s)
[losa] does not strip bte community (1.18 s)
[newy-re0] does not strip bte community (0.85 s)
[salt-re0] does not strip bte community (0.61 s)
[seat-re0] does not strip bte community (0.82 s)
[wash] does not strip bte community (0.82 s)


### No Martians Property

In [10]:
# Create the Batfish Session for the NoMartians tests
networkName = "NoMartians_Check_net"
snapshotName = "NoMartians_Check_snapshot"
bf = create(networkName,snapshotName,FULL_SNAPSHOT)

In [11]:
# Runs the NoMartians verification
def no_martians_at(l):
    return LocationPropertyPair(location=l,property=[Clause(prefixes=list(map(negate,MARTIANS)))])

target = no_martians_at(ROUTERS[0])
enforced = list(map(no_martians_at,ROUTERS[1:]))
internal_assumptions = list(map(no_martians_at,[outgoingEdgesFrom("64.57.28.251"), outgoingEdgesFrom("64.57.28.252")]))
query = IsolationQuery(target=target,assumptions=enforced + internal_assumptions)
t = MyTimer()
df = runIsolationVerificationQuestion(bf,query)
elapsed = t.stop()

print(f"Runtime: {elapsed} seconds")
print(f"Counterexample rows: {counterexamplesCount(df)}")

Runtime: 41.4 seconds
Counterexample rows: 0


In [12]:
# Displays the data frame with the verification details, this will be long because all invariants are displayed
show(df)

Location_Relevance  \
0             Target   
1         Assumption   
2  Internal Location   
3  Internal Location   
4  Internal Location   
5  Internal Location   

                                                                                                                                                                        Provided_Invariant  \
0  !prefix([198.18.0.0/15, 255.255.255.255/32, 0.0.0.0/0, 172.16.0.0/12, 192.88.99.1/32, 10.0.0.0/8, 127.0.0.0/8, 192.168.0.0/16, 169.254.0.0/16, 192.0.2.0/24, 224.0.0.0/4, 240.0.0.0/4])   
1                                                                                                                                                                                     true   
2                                                                                                                                                                                            
3                                                                                                                                                                                            
4  !prefix([198.18.0.0/15, 255.255.255.255/32, 0.0.0.0/0, 172.16.0.0/12, 192.88.99.1/32, 10.0.0.0/8, 127.0.0.0/8, 192.168.0.0/16, 169.254.0.0/16, 192.0.2.0/24, 224.0.0.0/4, 240.0.0.0/4])   
5                                                                                                                                                                                     true   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      